### Generate simulated animal health data for APHA Dashboard Project

In [ ]:
import pandas as pd 
import numpy as np 
from datetime import datetime, timedelta 
from dateutil.relativedelta import relativedelta
import random 


# Set seed for reproducibility
np.random.seed(42)


# Define date range (last 36 months)
end_date = datetime.now()

# Dynamically subtracts exactly 3 calender years (accounting for leap year - intead of timedelta)
start_date = end_date -  relativedelta(months=36)


# Define UK regions (Mapped to APHA's operational areas)
regions = [
    'North East (Newcastle)', 'North West (Preston/Carlisle)', 'Yorkshire & Humber (York)',
    'East Midlands (Crewe)', 'South West (Bristol/Exeter)', 'Wales (Cardiff/Carmarthen)', 
    'South East (Weybridge/Winchester)'
]

# Define animal types (relevant to APHA's surveillance)
apha_all_monitored_species = [
    'Cattle', 'Pigs', 'Sheep', 'Goats', 'Poultry', 'Gamebirds',
    'Horses', 'Alpacas', 'Llamas', 'Deer', 'Rabbits',
    'Wild birds', 'Bats', 'Badgers', 'Foxes', 'Amphibians', 'Marine mammals',
    'Dogs', 'Cats', 'Ferrets', 'Primates', 'Wild rodents'
]

diseases = {
    'Bovine TB': [
        'Cattle', 'Goats', 'Deer', 'Pigs', 'Sheep', 'Alpacas', 'Llamas', 
        'Badgers', 'Cats', 'Dogs'
    ],
    'Avian Influenza': [
        'Poultry', 'Gamebirds', 'Wild birds', 'Dogs', 'Cats', 'Foxes'
    ],
    'Foot and Mouth': [
        'Cattle', 'Sheep', 'Goats', 'Deer'
    ],
    'Bluetongue': [
        'Cattle', 'Sheep', 'Goats', 'Deer', 'Alpacas', 'Llamas'
    ],
    'African Swine Fever': [
        'Pigs', 'Wild boar'  # Strictly suids only per APHA statutory guidelines
    ],
    'Rabies': [
        'Dogs', 'Cats', 'Foxes', 'Bats', 'Badgers', 'Primates', 'Ferrets', 
        'Horses', 'Cattle', 'Sheep', 'Goats', 'Pigs', 'Deer', 'Wild rodents', 
        'Marine mammals', 'Alpacas', 'Llamas', 'Rabbits'  # Restricted strictly to mammals
    ],
    'Swine Dysentery': [
        'Pigs'
    ],
    'Scrapie': [
        'Sheep', 'Goats'
    ],
    'Newcastle Disease': [
        'Poultry', 'Gamebirds', 'Wild birds'  # Reflects real-world wild pigeon/waterfowl vectors
    ]                  
}

# Quick programmatic validation step
for disease, animals in diseases.items():
    invalid_animals = [a for a in animals if a not in apha_all_monitored_species and a != 'Wild boar']
    if invalid_animals:
        print(f"Warning: {disease} contains unlisted species tokens: {invalid_animals}")


# Define outbreak severity
severities = ['Low', 'Medium', 'High', 'Critical']

# Define laboratory result status
lab_results = ['Confirmed', 'Suspected', 'Negative', 'Pending']

# Generate 10,000 outbreak records
outbreaks = []

date_range_days = (end_date - start_date).days

for i in range(10000):
    date = start_date + timedelta(days=random.randint(0, date_range_days))
    region = random.choice(regions)
    disease = random.choice(list(diseases.keys()))

    # Ensure animal type is valid for the disease
    valid_animals = diseases[disease]
    animal = random.choice(valid_animals)
    severity = random.choices(severities, weights=[0.4, 0.3, 0.2, 0.1])[0]
    lab_result = random.choices(lab_results, weights=[0.6, 0.2, 0.1, 0.1])[0]
    animals_affected = random.randint(1, 1000)
    response_days = random.randint(1, 30)


    outbreaks.append({
        'outbreak_id': f'APH-{2023}-{i+1:04}',
        'date_reported': date.strftime('%Y-%m-%d'),
        'region': region,
        'disease': disease,
        'animal_type': animal,
        'severity': severity,
        'lab_result': lab_result,
        'animals_affected': animals_affected,
        'response_days': response_days
    })


df = pd.DataFrame(outbreaks)

# Validation: ensure generated data meets expectations
report_dates = pd.to_datetime(df['date_reported'])
assert report_dates.dt.date.min() >= start_date.date(), \
    f"Earliest date {report_dates.dt.date.min()} is before start_date {start_date.date()}"
assert report_dates.dt.date.max() <= end_date.date(), \
    f"Latest date {report_dates.dt.date.max()} is after end_date {end_date.date()}"
assert report_dates.dt.date.min() != report_dates.dt.date.max(), \
    "All outbreak dates are identical; date generation is not varying across the range"
assert df['severity'].isin(severities).all(), "Invalid severity values found"
assert df['lab_result'].isin(lab_results).all(), "Invalid lab_result values found"
valid_animals = {animal for animals in diseases.values() for animal in animals}
assert df['animal_type'].isin(valid_animals).all(), "Invalid animal_type values found"
assert len(df['outbreak_id']) == df['outbreak_id'].nunique(), "Duplicate outbreak_id values found"
print('Validation successful: generated outbreak data is within the expected range and format')

# Create a separate farm reference table (dimension table)
farms = []

for i in range(200):
    farms.append({
        'farm_id': f'FARM-{i+1:04d}',
        'region': random.choice(regions),
        'farm_type': random.choice(['Commercial', 'Smallholding', 'Organic', 'Research']),
        'size': random.choice(['Small (<50 animals)', 'Medium (50-200)', 'Large (200+)'])
    })


farm_df = pd.DataFrame(farms)


# Add farm_id to main outbreak data (linking fact to dimension)
df['farm_id'] = [random.choice(farm_df['farm_id'].tolist()) for _ in range(len(df))]

# Save to csv
df.to_csv('animal_health_outbreaks.csv', index=False)
farm_df.to_csv('farm_reference.csv', index=False)

print("Data generation successful\n")
print(f"Outbreak Records Created: {len(df)}\n")
print(f"Farm Records Created: {len(farm_df)}\n")
